# Amazon Bedrock Flows

A **Flow** is a visual/programmatic workflow that links steps - prompts, models, knowledge bases, Lambda functions, conditions - into an end-to-end pipeline. You define a graph of **nodes** connected by **connections**, deploy it, and invoke it with an input.

This notebook builds the playlist flow from the slides: given a `genre` and a `number`, it generates a playlist. It maps to the two slides - *create a flow programmatically* and *invoke a flow* - and fills in the node/connection detail the slides abbreviate with `...`.

> **Teaching/Learning Tip:** a Flow is orchestration you configure rather than code. For a single prompt it's overkill (just call `converse`), but the value shows when you chain many steps - retrieve from a knowledge base, branch on a condition, call a Lambda - without writing and hosting the glue code yourself.

## Setup: clients and an execution role

A flow runs under an IAM execution role that Bedrock assumes. Our flow only calls a model, so the role needs `bedrock:InvokeModel` on the prompt node's model, and a trust policy allowing `bedrock.amazonaws.com`.

In [ ]:
import json
import time
import boto3

REGION = "us-east-1"
acct = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
MODEL = f"arn:aws:bedrock:{REGION}:{acct}:inference-profile/us.amazon.nova-lite-v1:0"
ROLE_NAME = "BedrockFlowsDemoRole"

iam = boto3.client("iam")
agent = boto3.client("bedrock-agent", region_name=REGION)
runtime = boto3.client("bedrock-agent-runtime", region_name=REGION)

trust = {"Version": "2012-10-17", "Statement": [{
    "Effect": "Allow", "Principal": {"Service": "bedrock.amazonaws.com"},
    "Action": "sts:AssumeRole",
    "Condition": {"StringEquals": {"aws:SourceAccount": acct}}}]}
perms = {"Version": "2012-10-17", "Statement": [{
    "Effect": "Allow", "Action": ["bedrock:InvokeModel"],
    "Resource": [MODEL, f"arn:aws:bedrock:*::foundation-model/amazon.nova-lite-v1:0"]}]}

try:
    role_arn = iam.create_role(RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust))["Role"]["Arn"]
    print("Created role")
except iam.exceptions.EntityAlreadyExistsException:
    role_arn = iam.get_role(RoleName=ROLE_NAME)["Role"]["Arn"]
    print("Role exists")
iam.put_role_policy(RoleName=ROLE_NAME, PolicyName="FlowsInvoke",
    PolicyDocument=json.dumps(perms))
time.sleep(12)  # let the role propagate
print("role_arn:", role_arn)

## The flow definition: nodes and connections

This is the part the slide hides behind `...`. Three nodes:
- **FlowInput** - entry point; receives a JSON object (`genre`, `number`)
- **MakePlaylist** - a prompt node with an inline template using `{{genre}}` and `{{number}}`
- **FlowOutput** - returns the model's text

And three connections wire them: input's `document` feeds the prompt's `genre` and `number` inputs (extracted with the `$.data.genre` / `$.data.number` expressions), and the prompt's `modelCompletion` feeds the output.

> **Teaching/Learning Tip:** connections carry data between nodes, and the `expression` on an input pulls the right piece out of the upstream payload (JSONPath-style). That routing is the essence of a flow - nodes do work, connections move data.

In [ ]:
definition = {
    "nodes": [
        {"name": "FlowInput", "type": "Input",
         "outputs": [{"name": "document", "type": "Object"}]},
        {"name": "MakePlaylist", "type": "Prompt",
         "inputs": [
             {"name": "genre", "type": "String", "expression": "$.data.genre"},
             {"name": "number", "type": "Number", "expression": "$.data.number"},
         ],
         "outputs": [{"name": "modelCompletion", "type": "String"}],
         "configuration": {"prompt": {"sourceConfiguration": {"inline": {
             "modelId": MODEL, "templateType": "TEXT",
             "templateConfiguration": {"text": {
                 "text": "Make me a {{genre}} playlist consisting of the following number of songs: {{number}}.",
                 "inputVariables": [{"name": "genre"}, {"name": "number"}]}},
             "inferenceConfiguration": {"text": {"maxTokens": 300, "temperature": 0.7}},
         }}}}},
        {"name": "FlowOutput", "type": "Output",
         "inputs": [{"name": "document", "type": "String", "expression": "$.data"}]},
    ],
    "connections": [
        {"name": "in_to_genre", "source": "FlowInput", "target": "MakePlaylist", "type": "Data",
         "configuration": {"data": {"sourceOutput": "document", "targetInput": "genre"}}},
        {"name": "in_to_number", "source": "FlowInput", "target": "MakePlaylist", "type": "Data",
         "configuration": {"data": {"sourceOutput": "document", "targetInput": "number"}}},
        {"name": "prompt_to_out", "source": "MakePlaylist", "target": "FlowOutput", "type": "Data",
         "configuration": {"data": {"sourceOutput": "modelCompletion", "targetInput": "document"}}},
    ],
}
print("Flow definition built:", len(definition["nodes"]), "nodes,",
      len(definition["connections"]), "connections")

## Slide 1: create the flow programmatically

`create_flow` registers the definition. Then a flow must be **prepared** (compiled/validated) before it can run, and deployed via a **version** and an **alias** that you invoke.

> **Teaching/Learning Tip:** the create -> prepare -> version -> alias lifecycle mirrors how you'd promote code: build it, snapshot an immutable version, and point a stable alias (like `live`) at that version. You invoke the alias, so you can roll versions forward or back without changing the caller.

In [ ]:
flow = agent.create_flow(
    name="FlowCreatePlaylist",
    description="A flow that creates a playlist given a genre and number of songs.",
    executionRoleArn=role_arn,
    definition=definition,
)
flow_id = flow["id"]
print("flow_id:", flow_id)

# Prepare (compile) the flow and wait until it's ready
agent.prepare_flow(flowIdentifier=flow_id)
while True:
    status = agent.get_flow(flowIdentifier=flow_id)["status"]
    print("status:", status)
    if status in ("Prepared", "Failed"):
        break
    time.sleep(5)

# Snapshot a version and point an alias at it
version = agent.create_flow_version(flowIdentifier=flow_id)["version"]
alias = agent.create_flow_alias(flowIdentifier=flow_id, name="live",
    routingConfiguration=[{"flowVersion": version}])
alias_id = alias["id"]
print(f"version {version}, alias {alias_id}")

## Slide 2: invoke the flow

`invoke_flow` sends input to the flow's input node and streams back events. The result arrives as a stream we accumulate; a `flowCompletionEvent` tells us if it succeeded, and `flowOutputEvent` carries the output.

In [ ]:
response = runtime.invoke_flow(
    flowIdentifier=flow_id,
    flowAliasIdentifier=alias_id,
    inputs=[{
        "content": {"document": {"genre": "pop", "number": 3}},
        "nodeName": "FlowInput",
        "nodeOutputName": "document",
    }],
)

result = {}
for event in response.get("responseStream"):
    result.update(event)

if result["flowCompletionEvent"]["completionReason"] == "SUCCESS":
    print("Flow invocation was successful! Output:\n")
    print(result["flowOutputEvent"]["content"]["document"])
else:
    print("Flow did not succeed:", result["flowCompletionEvent"]["completionReason"])

## Cleanup

Delete in dependency order: alias, then flow (which removes its versions), then the IAM role.

In [ ]:
agent.delete_flow_alias(flowIdentifier=flow_id, aliasIdentifier=alias_id)
agent.delete_flow(flowIdentifier=flow_id)
iam.delete_role_policy(RoleName=ROLE_NAME, PolicyName="FlowsInvoke")
iam.delete_role(RoleName=ROLE_NAME)
print("Deleted flow, alias, and role.")